<a href="https://colab.research.google.com/github/jaiswalvik/MathematicalFoundationsOfGenerativeAI/blob/master/Assignment3_WGAN_BiGAN_DANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 3 — WGAN, WGAN-GP, BiGAN, and DANN

This notebook deliberately reuses the earlier notebook's data convention and preprocessing:
- `DATA_DIR/train/<class folders>` and `DATA_DIR/test/<class folders>`
- seeded, stratified 80/20 train/validation split from the training folder (`random_state=42`)
- RGB images, ImageNet mean/std normalization
- training transform: `RandomResizedCrop(224)` + horizontal flip
- evaluation transform: `Resize(256)` + `CenterCrop(224)`
- batch size 16 and 2 workers by default

It implements:
1. WGAN with weight clipping — MLP and DCGAN-style backbones
2. WGAN-GP — MLP and DCGAN-style backbones
3. BiGAN — encoder, generator, joint discriminator
4. DANN — feature extractor, label predictor, domain classifier, gradient reversal


## Runtime and reproducibility

In [1]:
# Colab users: select Runtime > Change runtime type > GPU.
# Install only the small optional metric dependency here; core models use PyTorch/torchvision.
%pip -q install scikit-learn matplotlib pandas pillow

In [2]:
import os, json, math, random, time, copy, itertools, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, utils
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


## 1. Mount Drive and configure paths

Keep `DATA_DIR` pointed at the **same dataset folder** used by Assignment 2. The expected layout is:

```text
dataset1/
  train/
    class_0/
    class_1/
    ...
  test/
    class_0/
    class_1/
    ...
```

If your dataset is a ZIP in Drive, set `ZIP_PATH` below. The split indices are regenerated deterministically with the same seed and stratification rule as Assignment 2.

In [3]:
# Google Colab only. In a local Jupyter environment, comment out these two lines
# and set DATA_DIR to your local dataset path.
from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = Path("/content/dataset1")
ZIP_PATH = Path("/content/drive/MyDrive/dataset1.zip")
if not (DATA_DIR / "train").is_dir() and ZIP_PATH.is_file():
    import zipfile
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall("/content")

OUTPUT_DIR = Path("/content/drive/MyDrive/assignment3_adversarial_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 2
LATENT_DIM = 100
EPOCHS_SMOKE = 1
# Suggested starting values after smoke test; tune based on GPU capacity.
WGAN_EPOCHS = 10
WGAN_GP_EPOCHS = 10
BIGAN_EPOCHS = 10
DANN_EPOCHS = 10

# WGAN hyperparameters
N_CRITIC = 5
WEIGHT_CLIP = 0.01
GP_LAMBDA = 10.0
WGAN_LR = 5e-5
WGAN_GP_LR = 1e-4

# DANN hyperparameters
DANN_LR = 1e-3
DANN_WEIGHT_DECAY = 1e-4

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"
assert TRAIN_DIR.is_dir(), f"Training folder not found: {TRAIN_DIR}. Set DATA_DIR or ZIP_PATH."
assert TEST_DIR.is_dir(), f"Test folder not found: {TEST_DIR}. Set DATA_DIR correctly."

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Dataset, transforms, and loaders

In [4]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_dataset_aug = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
train_dataset_eval = datasets.ImageFolder(TRAIN_DIR, transform=eval_transform)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=eval_transform)

class_names = train_dataset_aug.classes
num_classes = len(class_names)
class_to_idx = train_dataset_aug.class_to_idx
assert train_dataset_eval.class_to_idx == class_to_idx
assert test_dataset.class_to_idx == class_to_idx, "Train/test class folders must match."

targets = np.array(train_dataset_aug.targets)
indices = np.arange(len(targets))
train_idx, val_idx = train_test_split(
    indices, test_size=0.20, random_state=SEED, stratify=targets
)

train_subset = Subset(train_dataset_aug, train_idx)
val_subset = Subset(train_dataset_eval, val_idx)
test_subset = test_dataset

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
                          drop_last=True)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_subset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

print(f"Classes ({num_classes}):", class_names)
print(f"Train={len(train_subset)}, Val={len(val_subset)}, Test={len(test_subset)}")
assert len(train_loader) > 0, "No full training batch. Reduce BATCH_SIZE."

Classes (10): ['n01820546', 'n01833805', 'n01843065', 'n01843383', 'n01855032', 'n01855672', 'n01860187', 'n01872401', 'n01882714', 'n02012849']
Train=9872, Val=2469, Test=500


### 2.1 Image scaling convention

The Assignment 2 notebook used ImageNet-normalized real images, while generators emit `tanh` images in `[-1,1]`. We preserve that convention: generated images are mapped to `[0,1]` and then ImageNet-normalized before entering critics/discriminators. For display and image metrics, tensors are converted back to `[0,1]`.

In [5]:
def _normalization_tensors(x):
    mean = torch.as_tensor(
        IMAGENET_MEAN, dtype=x.dtype, device=x.device
    ).view(1, 3, 1, 1)

    std = torch.as_tensor(
        IMAGENET_STD, dtype=x.dtype, device=x.device
    ).view(1, 3, 1, 1)

    return mean, std


def generator_to_model_scale(fake_tanh):
    mean, std = _normalization_tensors(fake_tanh)

    rgb01 = ((fake_tanh + 1.0) / 2.0).clamp(0, 1)

    return (rgb01 - mean) / std

def fake_to_display_scale(fake_tanh):
    return ((fake_tanh + 1.0) / 2.0).clamp(0, 1)

def show_tensor_grid(images, title="", nrow=8, normalize=False, value_range=None):
    grid = utils.make_grid(images.detach().cpu(), nrow=nrow,
                           normalize=normalize, value_range=value_range)
    plt.figure(figsize=(10, 8))
    plt.imshow(grid.permute(1, 2, 0).clamp(0, 1))
    plt.axis("off")
    plt.title(title)
    plt.show()

real_batch, _ = next(iter(train_loader))
show_tensor_grid(real_to_display_scale(real_batch[:16]), "Real training images", nrow=4)

NameError: name 'real_to_display_scale' is not defined

## 3. Shared generator and critic backbones

The MLP version keeps a fully connected generator/critic core while using a compact 64×64 generated canvas that is resized to the configured image resolution. This avoids a prohibitively large fully connected layer at 224×224. The DCGAN-style version uses transposed convolutions for generation and strided convolutions for criticism.

**WGAN critics output unrestricted real-valued scores. They must not use a sigmoid.**

In [ ]:
def init_dcgan_weights(module):
    if isinstance(module, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
        nn.init.normal_(module.weight, 0.0, 0.02)
        if module.bias is not None:
            nn.init.zeros_(module.bias)

class MLPGenerator(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, channels=3, image_size=IMAGE_SIZE, canvas=64):
        super().__init__()
        self.channels, self.image_size, self.canvas = channels, image_size, canvas
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 512), nn.BatchNorm1d(512), nn.ReLU(True),
            nn.Linear(512, 1024), nn.BatchNorm1d(1024), nn.ReLU(True),
            nn.Linear(1024, channels * canvas * canvas), nn.Tanh()
        )
    def forward(self, z):
        x = self.net(z).view(z.size(0), self.channels, self.canvas, self.canvas)
        if self.image_size != self.canvas:
            x = F.interpolate(x, size=(self.image_size, self.image_size),
                              mode="bilinear", align_corners=False)
        return x

class MLPCritic(nn.Module):
    def __init__(self, channels=3, image_size=IMAGE_SIZE, canvas=32):
        super().__init__()
        self.canvas = canvas
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels * canvas * canvas, 1024), nn.LeakyReLU(0.2, True),
            nn.Linear(1024, 512), nn.LeakyReLU(0.2, True),
            nn.Linear(512, 1)
        )
    def forward(self, x):
        x = F.adaptive_avg_pool2d(x, (self.canvas, self.canvas))
        return self.net(x)

class DCGANGenerator(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, channels=3, image_size=IMAGE_SIZE, base=64):
        super().__init__()
        self.image_size = image_size
        start = 7
        stages = max(1, int(math.ceil(math.log2(image_size / start))))
        first = base * 8
        self.fc = nn.Sequential(
            nn.Linear(latent_dim, first * start * start),
            nn.BatchNorm1d(first * start * start), nn.ReLU(True)
        )
        schedule = [base*4, base*2, base, max(base//2, 8), max(base//4, 8)]
        layers, in_ch = [], first
        for s in range(stages):
            out_ch = schedule[min(s, len(schedule)-1)]
            layers += [nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1, bias=False),
                       nn.BatchNorm2d(out_ch), nn.ReLU(True)]
            in_ch = out_ch
        layers += [nn.Conv2d(in_ch, channels, 3, 1, 1), nn.Tanh()]
        self.net = nn.Sequential(*layers)
        self.start = start
    def forward(self, z):
        x = self.fc(z).view(z.size(0), -1, self.start, self.start)
        x = self.net(x)
        if x.shape[-1] != self.image_size:
            x = F.interpolate(x, size=(self.image_size, self.image_size),
                              mode="bilinear", align_corners=False)
        return x

class DCGANCritic(nn.Module):
    def __init__(self, channels=3, base=64):
        super().__init__()
        layers = [
            nn.Conv2d(channels, base, 4, 2, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(base, base*2, 4, 2, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(base*2, base*4, 4, 2, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(base*4, base*8, 4, 2, 1), nn.LeakyReLU(0.2, True),
            nn.AdaptiveAvgPool2d(1)
        ]
        self.features = nn.Sequential(*layers)
        self.score = nn.Linear(base*8, 1)
    def forward(self, x):
        return self.score(self.features(x).flatten(1))

def make_gan_backbone(kind):
    if kind == "mlp":
        return MLPGenerator().to(device), MLPCritic().to(device)
    if kind == "dcgan":
        G, C = DCGANGenerator().to(device), DCGANCritic().to(device)
        G.apply(init_dcgan_weights); C.apply(init_dcgan_weights)
        return G, C
    raise ValueError("kind must be 'mlp' or 'dcgan'")

## 4. WGAN and WGAN-GP objectives

For WGAN, the critic is trained to maximize `mean(C(real)) - mean(C(fake))`, while the generator minimizes `-mean(C(fake))`. Weight clipping constrains critic parameters.

For WGAN-GP, weight clipping is replaced by a gradient penalty:
`lambda * E[(||grad_xhat C(xhat)||_2 - 1)^2]`.
The interpolation `xhat` is sampled between real and generated images. The critic contains no sigmoid and no batch normalization, because batch-dependent operations interfere with the per-example gradient penalty.

In [ ]:
def gradient_penalty(critic, real, fake):
    batch_size = real.size(0)
    alpha = torch.rand(batch_size, 1, 1, 1, device=real.device)
    interpolated = (alpha * real + (1 - alpha) * fake).requires_grad_(True)
    scores = critic(interpolated)
    grads = torch.autograd.grad(
        outputs=scores, inputs=interpolated,
        grad_outputs=torch.ones_like(scores),
        create_graph=True, retain_graph=True, only_inputs=True
    )[0]
    grads = grads.view(batch_size, -1)
    return ((grads.norm(2, dim=1) - 1) ** 2).mean()

def save_generated_grid(G, name, epoch, fixed_noise):
    G.eval()
    with torch.no_grad():
        fake = fake_to_display_scale(G(fixed_noise))
    path = OUTPUT_DIR / f"{name}_epoch_{epoch:03d}.png"
    grid = utils.make_grid(fake.cpu(), nrow=8)
    plt.figure(figsize=(10, 8))
    plt.imshow(grid.permute(1, 2, 0).clamp(0, 1))
    plt.axis("off"); plt.title(f"{name} — epoch {epoch}")
    plt.savefig(path, bbox_inches="tight"); plt.show(); plt.close()
    G.train()
    return str(path)

fixed_noise = torch.randn(64, LATENT_DIM, device=device)

## 5. WGAN training loop (weight clipping)

In [ ]:
def train_wgan(kind="mlp", epochs=EPOCHS_SMOKE, n_critic=N_CRITIC,
               clip_value=WEIGHT_CLIP, sample_every=1):
    G, C = make_gan_backbone(kind)
    opt_G = optim.RMSprop(G.parameters(), lr=WGAN_LR)
    opt_C = optim.RMSprop(C.parameters(), lr=WGAN_LR)
    history = {"critic_loss": [], "generator_loss": []}
    global_step = 0

    for epoch in range(1, epochs + 1):
        G.train(); C.train()
        c_running, g_running, c_count, g_count = 0.0, 0.0, 0, 0
        for real, _ in train_loader:
            real = real.to(device)
            bs = real.size(0)
            z = torch.randn(bs, LATENT_DIM, device=device)
            fake = generator_to_model_scale(G(z)).detach()

            opt_C.zero_grad(set_to_none=True)
            c_loss = C(fake).mean() - C(real).mean()
            c_loss.backward()
            opt_C.step()
            for p in C.parameters():
                p.data.clamp_(-clip_value, clip_value)
            c_running += c_loss.item(); c_count += 1

            if global_step % n_critic == 0:
                z = torch.randn(bs, LATENT_DIM, device=device)
                opt_G.zero_grad(set_to_none=True)
                g_loss = -C(generator_to_model_scale(G(z))).mean()
                g_loss.backward(); opt_G.step()
                g_running += g_loss.item(); g_count += 1
            global_step += 1

        history["critic_loss"].append(c_running / max(c_count, 1))
        history["generator_loss"].append(g_running / max(g_count, 1))
        print(f"[WGAN-{kind}] epoch {epoch}/{epochs} | "
              f"C={history['critic_loss'][-1]:.4f} | G={history['generator_loss'][-1]:.4f}")
        if epoch % sample_every == 0:
            save_generated_grid(G, f"wgan_{kind}", epoch, fixed_noise)

    torch.save({"G": G.state_dict(), "critic": C.state_dict(),
                "history": history, "kind": kind},
               OUTPUT_DIR / f"wgan_{kind}.pt")
    return G, C, history

def plot_wgan_history(history, title):
    plt.figure(figsize=(8, 4))
    plt.plot(history["critic_loss"], label="Critic loss")
    plt.plot(history["generator_loss"], label="Generator loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title(title); plt.legend(); plt.show()

In [ ]:
# Run both required WGAN backbones.
wgan_mlp_G, wgan_mlp_C, wgan_mlp_history = train_wgan("mlp", epochs=EPOCHS_SMOKE)
plot_wgan_history(wgan_mlp_history, "WGAN — MLP")

wgan_dcgan_G, wgan_dcgan_C, wgan_dcgan_history = train_wgan("dcgan", epochs=EPOCHS_SMOKE)
plot_wgan_history(wgan_dcgan_history, "WGAN — DCGAN-style")

## 6. WGAN-GP training loop (gradient penalty)

In [ ]:
def train_wgan_gp(kind="mlp", epochs=EPOCHS_SMOKE, n_critic=N_CRITIC,
                  gp_lambda=GP_LAMBDA, sample_every=1):
    G, C = make_gan_backbone(kind)
    opt_G = optim.Adam(G.parameters(), lr=WGAN_GP_LR, betas=(0.0, 0.9))
    opt_C = optim.Adam(C.parameters(), lr=WGAN_GP_LR, betas=(0.0, 0.9))
    history = {"critic_loss": [], "generator_loss": [], "gradient_penalty": []}
    global_step = 0

    for epoch in range(1, epochs + 1):
        G.train(); C.train()
        c_running = g_running = gp_running = 0.0
        c_count = g_count = 0
        for real, _ in train_loader:
            real = real.to(device)
            bs = real.size(0)
            z = torch.randn(bs, LATENT_DIM, device=device)
            fake = generator_to_model_scale(G(z)).detach()

            opt_C.zero_grad(set_to_none=True)
            real_score = C(real).mean()
            fake_score = C(fake).mean()
            gp = gradient_penalty(C, real, fake)
            c_loss = fake_score - real_score + gp_lambda * gp
            c_loss.backward(); opt_C.step()
            c_running += c_loss.item()
            gp_running += gp.item()
            c_count += 1

            if global_step % n_critic == 0:
                z = torch.randn(bs, LATENT_DIM, device=device)
                opt_G.zero_grad(set_to_none=True)
                g_loss = -C(generator_to_model_scale(G(z))).mean()
                g_loss.backward(); opt_G.step()
                g_running += g_loss.item(); g_count += 1
            global_step += 1

        history["critic_loss"].append(c_running / max(c_count, 1))
        history["generator_loss"].append(g_running / max(g_count, 1))
        history["gradient_penalty"].append(gp_running / max(c_count, 1))
        print(f"[WGAN-GP-{kind}] epoch {epoch}/{epochs} | "
              f"C={history['critic_loss'][-1]:.4f} | G={history['generator_loss'][-1]:.4f} | "
              f"GP={history['gradient_penalty'][-1]:.4f}")
        if epoch % sample_every == 0:
            save_generated_grid(G, f"wgan_gp_{kind}", epoch, fixed_noise)

    torch.save({"G": G.state_dict(), "critic": C.state_dict(),
                "history": history, "kind": kind},
               OUTPUT_DIR / f"wgan_gp_{kind}.pt")
    return G, C, history

In [ ]:
wgan_gp_mlp_G, wgan_gp_mlp_C, wgan_gp_mlp_history = train_wgan_gp("mlp", epochs=EPOCHS_SMOKE)
plot_wgan_history(wgan_gp_mlp_history, "WGAN-GP — MLP")

wgan_gp_dcgan_G, wgan_gp_dcgan_C, wgan_gp_dcgan_history = train_wgan_gp("dcgan", epochs=EPOCHS_SMOKE)
plot_wgan_history(wgan_gp_dcgan_history, "WGAN-GP — DCGAN-style")

## 7. BiGAN: encoder + generator + joint discriminator

The BiGAN discriminator receives a pair `(x,z)` and predicts whether it came from:
- the **data/encoder joint**: `(x, E(x))`
- the **generator/prior joint**: `(G(z), z)`

The encoder and generator are optimized together to fool the joint discriminator. The optional reconstruction diagnostic `G(E(x))` is visualized; standard BiGAN adversarial training does not add a pixel reconstruction loss unless explicitly chosen as an extension.

In [ ]:
class BiGANEncoder(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, channels=3, base=64):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(channels, base, 4, 2, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(base, base*2, 4, 2, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(base*2, base*4, 4, 2, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(base*4, base*8, 4, 2, 1), nn.LeakyReLU(0.2, True),
            nn.AdaptiveAvgPool2d(1)
        )
        self.to_z = nn.Linear(base*8, latent_dim)
    def forward(self, x):
        return self.to_z(self.features(x).flatten(1))

class JointDiscriminator(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, channels=3, base=64):
        super().__init__()
        self.x_features = nn.Sequential(
            nn.Conv2d(channels, base, 4, 2, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(base, base*2, 4, 2, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(base*2, base*4, 4, 2, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(base*4, base*8, 4, 2, 1), nn.LeakyReLU(0.2, True),
            nn.AdaptiveAvgPool2d(1)
        )
        self.joint = nn.Sequential(
            nn.Linear(base*8 + latent_dim, 512), nn.LeakyReLU(0.2, True),
            nn.Linear(512, 1)
        )
    def forward(self, x, z):
        h = self.x_features(x).flatten(1)
        return self.joint(torch.cat([h, z], dim=1))

def train_bigan(epochs=EPOCHS_SMOKE, lr=2e-4):
    G = DCGANGenerator().to(device)
    E = BiGANEncoder().to(device)
    D = JointDiscriminator().to(device)
    G.apply(init_dcgan_weights); E.apply(init_dcgan_weights); D.apply(init_dcgan_weights)
    opt_D = optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_EG = optim.Adam(itertools.chain(E.parameters(), G.parameters()),
                        lr=lr, betas=(0.5, 0.999))
    bce = nn.BCEWithLogitsLoss()
    history = {"d_loss": [], "eg_loss": []}

    for epoch in range(1, epochs+1):
        D.train(); E.train(); G.train()
        d_sum = eg_sum = 0.0; n = 0
        for real, _ in train_loader:
            real = real.to(device); bs = real.size(0)
            z_prior = torch.randn(bs, LATENT_DIM, device=device)

            # Discriminator: data/encoder pairs are real; generator/prior pairs are fake.
            with torch.no_grad():
                z_encoded = E(real)
                fake_tanh = G(z_prior)
                fake = generator_to_model_scale(fake_tanh)
            opt_D.zero_grad(set_to_none=True)
            d_real = D(real, z_encoded)
            d_fake = D(fake, z_prior)
            d_loss = bce(d_real, torch.ones_like(d_real)) + bce(d_fake, torch.zeros_like(d_fake))
            d_loss.backward(); opt_D.step()

            # Encoder + generator fool the joint discriminator.
            z_prior = torch.randn(bs, LATENT_DIM, device=device)
            opt_EG.zero_grad(set_to_none=True)
            z_encoded = E(real)
            fake = generator_to_model_scale(G(z_prior))
            eg_loss = bce(D(fake, z_prior), torch.ones(bs, 1, device=device)) +                       bce(D(real, z_encoded), torch.zeros(bs, 1, device=device))
            eg_loss.backward(); opt_EG.step()

            d_sum += d_loss.item(); eg_sum += eg_loss.item(); n += 1

        history["d_loss"].append(d_sum/max(n,1))
        history["eg_loss"].append(eg_sum/max(n,1))
        print(f"[BiGAN] epoch {epoch}/{epochs} | D={history['d_loss'][-1]:.4f} | "
              f"E+G={history['eg_loss'][-1]:.4f}")

    torch.save({"G": G.state_dict(), "E": E.state_dict(), "D": D.state_dict(),
                "history": history}, OUTPUT_DIR/"bigan.pt")
    return G, E, D, history

bigan_G, bigan_E, bigan_D, bigan_history = train_bigan(epochs=EPOCHS_SMOKE)

In [ ]:
plt.figure(figsize=(8,4))
plt.plot(bigan_history["d_loss"], label="Joint discriminator")
plt.plot(bigan_history["eg_loss"], label="Encoder + generator")
plt.xlabel("Epoch"); plt.ylabel("BCE loss"); plt.title("BiGAN training losses")
plt.legend(); plt.show()

@torch.no_grad()
def show_bigan_reconstructions(G, E, loader=val_loader, n=8):
    G.eval(); E.eval()
    real, _ = next(iter(loader))
    real = real[:n].to(device)
    z = E(real)
    recon = fake_to_display_scale(G(z))
    originals = real_to_display_scale(real)
    comparison = torch.cat([originals, recon], dim=0)
    show_tensor_grid(comparison, "BiGAN: top = original, bottom = G(E(x))", nrow=n)
    G.train(); E.train()

show_bigan_reconstructions(bigan_G, bigan_E)

### 7.1 BiGAN latent interpolation

In [ ]:
@torch.no_grad()
def show_bigan_interpolation(G, steps=8):
    G.eval()
    z1 = torch.randn(1, LATENT_DIM, device=device)
    z2 = torch.randn(1, LATENT_DIM, device=device)
    alphas = torch.linspace(0, 1, steps, device=device)
    z = torch.cat([(1-a)*z1 + a*z2 for a in alphas], dim=0)
    show_tensor_grid(fake_to_display_scale(G(z)), "BiGAN latent interpolation", nrow=steps)
    G.train()

show_bigan_interpolation(bigan_G)

## 8. DANN: unsupervised domain adaptation

The source and target domains below are created from the **same training images** using different image transformations, which is one of the example domain definitions suggested by the assignment. Source labels are used by the label classifier. Target labels are intentionally not used in the training loss; they are used only for final target-domain test evaluation.

- Source domain: Assignment 2 training augmentation
- Target domain: stronger colour/contrast augmentation
- Test domain: deterministic evaluation transform plus the same target-style colour transform

If your actual task defines source/target as different subdomains, replace the target transform and datasets below while preserving the same training/validation/test split protocol.

In [ ]:
target_domain_train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.35, hue=0.08),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
class DeterministicTargetStyle:
    # Fixed PIL adjustments make target-domain test evaluation reproducible.
    def __call__(self, image):
        from PIL import ImageEnhance
        image = ImageEnhance.Color(image).enhance(0.75)
        image = ImageEnhance.Contrast(image).enhance(1.20)
        image = ImageEnhance.Brightness(image).enhance(1.10)
        return image

target_domain_eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    DeterministicTargetStyle(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

source_train_full = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
target_train_full = datasets.ImageFolder(TRAIN_DIR, transform=target_domain_train_transform)
target_test_dataset = datasets.ImageFolder(TEST_DIR, transform=target_domain_eval_transform)

source_train_loader = DataLoader(Subset(source_train_full, train_idx), batch_size=BATCH_SIZE,
                                 shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
target_train_loader = DataLoader(Subset(target_train_full, train_idx), batch_size=BATCH_SIZE,
                                 shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
target_test_loader = DataLoader(target_test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                                num_workers=NUM_WORKERS)

class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None

def grad_reverse(x, lambd=1.0):
    return GradientReversalFunction.apply(x, lambd)

class DANNFeatureExtractor(nn.Module):
    def __init__(self, channels=3, base=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, base, 5, 2, 2), nn.BatchNorm2d(base), nn.ReLU(True),
            nn.Conv2d(base, base*2, 5, 2, 2), nn.BatchNorm2d(base*2), nn.ReLU(True),
            nn.Conv2d(base*2, base*4, 3, 2, 1), nn.BatchNorm2d(base*4), nn.ReLU(True),
            nn.Conv2d(base*4, base*4, 3, 2, 1), nn.BatchNorm2d(base*4), nn.ReLU(True),
            nn.AdaptiveAvgPool2d(1), nn.Flatten()
        )
        self.output_dim = base*4
    def forward(self, x):
        return self.net(x)

class DANNLabelClassifier(nn.Module):
    def __init__(self, feature_dim, n_classes):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(feature_dim, 256), nn.ReLU(True),
                                 nn.Dropout(0.3), nn.Linear(256, n_classes))
    def forward(self, f):
        return self.net(f)

class DANNDomainClassifier(nn.Module):
    def __init__(self, feature_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(feature_dim, 128), nn.ReLU(True),
                                 nn.Dropout(0.3), nn.Linear(128, 2))
    def forward(self, f):
        return self.net(f)

In [ ]:
def dann_lambda(progress):
    # Smooth ramp from 0 toward 1, as commonly used in DANN.
    return 2.0 / (1.0 + math.exp(-10.0 * progress)) - 1.0

@torch.no_grad()
def evaluate_classifier(Fnet, Cnet, loader):
    Fnet.eval(); Cnet.eval()
    ys, preds = [], []
    for x, y in loader:
        x = x.to(device)
        pred = Cnet(Fnet(x)).argmax(1).cpu().numpy()
        preds.extend(pred); ys.extend(y.numpy())
    Fnet.train(); Cnet.train()
    return accuracy_score(ys, preds), np.asarray(ys), np.asarray(preds)

def train_dann(epochs=EPOCHS_SMOKE):
    Fnet = DANNFeatureExtractor().to(device)
    Cnet = DANNLabelClassifier(Fnet.output_dim, num_classes).to(device)
    Dnet = DANNDomainClassifier(Fnet.output_dim).to(device)
    params = list(Fnet.parameters()) + list(Cnet.parameters()) + list(Dnet.parameters())
    optimizer = optim.Adam(params, lr=DANN_LR, weight_decay=DANN_WEIGHT_DECAY)
    cls_loss_fn = nn.CrossEntropyLoss()
    dom_loss_fn = nn.CrossEntropyLoss()
    history = {"total_loss": [], "class_loss": [], "domain_loss": [],
               "source_train_accuracy": [], "target_test_accuracy": []}

    steps_per_epoch = min(len(source_train_loader), len(target_train_loader))
    for epoch in range(1, epochs+1):
        Fnet.train(); Cnet.train(); Dnet.train()
        src_iter, tgt_iter = iter(source_train_loader), iter(target_train_loader)
        sums = np.zeros(3, dtype=float); correct = total = 0
        for step in range(steps_per_epoch):
            xs, ys = next(src_iter)
            xt, _target_labels_ignored = next(tgt_iter)
            xs, ys, xt = xs.to(device), ys.to(device), xt.to(device)
            progress = ((epoch-1)*steps_per_epoch + step) / max(epochs*steps_per_epoch-1, 1)
            lambd = dann_lambda(progress)

            optimizer.zero_grad(set_to_none=True)
            fs, ft = Fnet(xs), Fnet(xt)
            class_logits = Cnet(fs)
            class_loss = cls_loss_fn(class_logits, ys)

            domain_features = torch.cat([fs, ft], dim=0)
            domain_labels = torch.cat([
                torch.zeros(xs.size(0), dtype=torch.long, device=device),
                torch.ones(xt.size(0), dtype=torch.long, device=device)
            ])
            domain_logits = Dnet(grad_reverse(domain_features, lambd))
            domain_loss = dom_loss_fn(domain_logits, domain_labels)
            total_loss = class_loss + domain_loss
            total_loss.backward(); optimizer.step()

            sums += [total_loss.item(), class_loss.item(), domain_loss.item()]
            correct += (class_logits.argmax(1) == ys).sum().item()
            total += ys.size(0)

        source_acc = correct / max(total, 1)
        target_acc, _, _ = evaluate_classifier(Fnet, Cnet, target_test_loader)
        means = sums / max(steps_per_epoch, 1)
        history["total_loss"].append(means[0])
        history["class_loss"].append(means[1])
        history["domain_loss"].append(means[2])
        history["source_train_accuracy"].append(source_acc)
        history["target_test_accuracy"].append(target_acc)
        print(f"[DANN] epoch {epoch}/{epochs} | total={means[0]:.4f} "
              f"class={means[1]:.4f} domain={means[2]:.4f} "
              f"source train acc={source_acc:.4f} target test acc={target_acc:.4f}")

    torch.save({"F": Fnet.state_dict(), "C": Cnet.state_dict(),
                "D": Dnet.state_dict(), "history": history,
                "class_names": class_names}, OUTPUT_DIR/"dann.pt")
    return Fnet, Cnet, Dnet, history

dann_F, dann_C, dann_D, dann_history = train_dann(epochs=EPOCHS_SMOKE)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(dann_history["total_loss"], label="Total")
axes[0].plot(dann_history["class_loss"], label="Classification")
axes[0].plot(dann_history["domain_loss"], label="Domain")
axes[0].set_title("DANN losses"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].plot(dann_history["source_train_accuracy"], label="Source train accuracy")
axes[1].plot(dann_history["target_test_accuracy"], label="Target test accuracy")
axes[1].set_title("DANN accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.tight_layout(); plt.show()

target_acc, y_true, y_pred = evaluate_classifier(dann_F, dann_C, target_test_loader)
print("Final target-domain test accuracy:", round(target_acc, 4))
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(max(6, num_classes*0.6), max(5, num_classes*0.5)))
plt.imshow(cm, interpolation="nearest")
plt.title("DANN target-domain test confusion matrix")
plt.xlabel("Predicted"); plt.ylabel("True")
plt.xticks(range(num_classes), class_names, rotation=90)
plt.yticks(range(num_classes), class_names)
plt.colorbar(); plt.tight_layout(); plt.show()

## 9. Generative-model evaluation

The assignment requests FID/KID/IS where possible. These metrics require an Inception feature extractor and may need additional downloads/dependencies. The cell below attempts FID and KID through `torchmetrics`; if unavailable, it reports a clearly labelled nearest-neighbour pixel-space proxy. **The proxy is not FID/KID/IS.**

Use the held-out test images for real-image evaluation. Generated images are sampled independently from the prior.

In [ ]:
@torch.no_grad()
def generate_display_images(G, n=256, batch_size=BATCH_SIZE):
    G.eval(); chunks=[]; remaining=n
    while remaining:
        bs=min(batch_size, remaining)
        chunks.append(fake_to_display_scale(G(torch.randn(bs, LATENT_DIM, device=device))).cpu())
        remaining-=bs
    G.train()
    return torch.cat(chunks)

@torch.no_grad()
def collect_display_images(loader, n=256):
    chunks=[]; remaining=n
    for x, _ in loader:
        bs=min(x.size(0), remaining)
        chunks.append(real_to_display_scale(x[:bs].to(device)).cpu())
        remaining-=bs
        if remaining<=0: break
    return torch.cat(chunks)

def nearest_neighbor_proxy(real_imgs, fake_imgs, side=16):
    real=F.interpolate(real_imgs, size=(side,side), mode="bilinear", align_corners=False).flatten(1)
    fake=F.interpolate(fake_imgs, size=(side,side), mode="bilinear", align_corners=False).flatten(1)
    real=F.normalize(real, dim=1); fake=F.normalize(fake, dim=1)
    distances=1.0 - fake @ real.T
    return float(distances.min(dim=1).values.mean())

def try_fid_kid(real_imgs, fake_imgs):
    try:
        from torchmetrics.image.fid import FrechetInceptionDistance
        from torchmetrics.image.kid import KernelInceptionDistance
        fid = FrechetInceptionDistance(feature=64, normalize=True).to(device)
        kid = KernelInceptionDistance(subset_size=min(50, len(real_imgs), len(fake_imgs)),
                                      normalize=True).to(device)
        for batch in real_imgs.split(32):
            fid.update(batch.to(device), real=True); kid.update(batch.to(device), real=True)
        for batch in fake_imgs.split(32):
            fid.update(batch.to(device), real=False); kid.update(batch.to(device), real=False)
        return {"FID": float(fid.compute().cpu()), "KID_mean": float(kid.compute()[0].cpu()),
                "metric_note": "TorchMetrics Inception metrics"}
    except Exception as exc:
        return {"NN_proxy": nearest_neighbor_proxy(real_imgs, fake_imgs),
                "metric_note": f"Proxy only; FID/KID unavailable: {type(exc).__name__}: {exc}"}

real_eval = collect_display_images(test_loader, n=min(256, len(test_subset)))

In [ ]:
# Evaluate each trained generator on the same number of test images.
generators = {
    "WGAN-MLP": wgan_mlp_G,
    "WGAN-DCGAN": wgan_dcgan_G,
    "WGAN-GP-MLP": wgan_gp_mlp_G,
    "WGAN-GP-DCGAN": wgan_gp_dcgan_G,
    "BiGAN": bigan_G,
}
metric_rows = []
for name, gen in generators.items():
    fake_eval = generate_display_images(gen, n=len(real_eval))
    result = try_fid_kid(real_eval, fake_eval)
    metric_rows.append({"Model": name, **result})
    show_tensor_grid(fake_eval[:16], f"{name}: generated test-size sample", nrow=4)
metric_table = pd.DataFrame(metric_rows)
display(metric_table)
metric_table.to_csv(OUTPUT_DIR/"generative_metrics.csv", index=False)

## 10. Consolidated results and artifacts

This section writes measured results to the output folder. Do not report smoke-test results as final training results: increase epochs, rerun, and record the actual configuration used.

In [ ]:
summary = {
    "dataset_dir": str(DATA_DIR),
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "seed": SEED,
    "class_names": class_names,
    "split": {"train": len(train_subset), "validation": len(val_subset), "test": len(test_subset)},
    "epochs": {"smoke": EPOCHS_SMOKE, "wgan": WGAN_EPOCHS,
               "wgan_gp": WGAN_GP_EPOCHS, "bigan": BIGAN_EPOCHS, "dann": DANN_EPOCHS},
    "dann_final_target_test_accuracy": float(target_acc),
    "notes": [
        "WGAN critic scores are unrestricted real values; no sigmoid.",
        "WGAN uses weight clipping; WGAN-GP uses gradient penalty instead of clipping.",
        "DANN target labels are not used in training; target test labels are used only for evaluation.",
        "If NN_proxy is reported, it is not FID/KID/IS."
    ]
}
with open(OUTPUT_DIR/"assignment3_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("Saved outputs to:", OUTPUT_DIR)
print(sorted(p.name for p in OUTPUT_DIR.iterdir()))